# Projeto Airbnb Rio - Ferramenta de Previsão de Preço de Imóvel para pessoas comuns 

### Contexto

No Airbnb, qualquer pessoa que tenha um quarto ou um imóvel de qualquer tipo (apartamento, casa, chalé, pousada, etc.) pode ofertar o seu imóvel para ser alugado por diária.

Você cria o seu perfil de host (pessoa que disponibiliza um imóvel para aluguel por diária) e cria o anúncio do seu imóvel.

Nesse anúncio, o host deve descrever as características do imóvel da forma mais completa possível, de forma a ajudar os locadores/viajantes a escolherem o melhor imóvel para eles (e de forma a tornar o seu anúncio mais atrativo)

Existem dezenas de personalizações possíveis no seu anúncio, desde quantidade mínima de diária, preço, quantidade de quartos, até regras de cancelamento, taxa extra para hóspedes extras, exigência de verificação de identidade do locador, etc.

### Nosso objetivo

Construir um modelo de previsão de preço que permita uma pessoa comum que possui um imóvel possa saber quanto deve cobrar pela diária do seu imóvel.

Ou ainda, para o locador comum, dado o imóvel que ele está buscando, ajudar a saber se aquele imóvel está com preço atrativo (abaixo da média para imóveis com as mesmas características) ou não.

### O que temos disponível, inspirações e créditos

As bases de dados foram retiradas do site kaggle: https://www.kaggle.com/allanbruno/airbnb-rio-de-janeiro

Elas estão disponíveis para download abaixo da aula (se você puxar os dados direto do Kaggle pode ser que encontre resultados diferentes dos meus, afinal as bases de dados podem ter sido atualizadas).

Caso queira uma outra solução, podemos olhar como referência a solução do usuário Allan Bruno do kaggle no Notebook: https://www.kaggle.com/allanbruno/helping-regular-people-price-listings-on-airbnb

Você vai perceber semelhanças entre a solução que vamos desenvolver aqui e a dele, mas também algumas diferenças significativas no processo de construção do projeto.

- As bases de dados são os preços dos imóveis obtidos e suas respectivas características em cada mês.
- Os preços são dados em reais (R$)
- Temos bases de abril de 2018 a maio de 2020, com exceção de junho de 2018 que não possui base de dados

### Expectativas Iniciais

- Acredito que a sazonalidade pode ser um fator importante, visto que meses como dezembro costumam ser bem caros no RJ
- A localização do imóvel deve fazer muita diferença no preço, já que no Rio de Janeiro a localização pode mudar completamente as características do lugar (segurança, beleza natural, pontos turísticos)
- Adicionais/Comodidades podem ter um impacto significativo, visto que temos muitos prédios e casas antigos no Rio de Janeiro

Vamos descobrir o quanto esses fatores impactam e se temos outros fatores não tão intuitivos que são extremamente importantes.

### Importar Bibliotecas e Bases de Dados

In [ ]:
import pandas as pd
import pathlib 
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.model_selection import train_test_split
import time

### Consolidar Base de Dados

In [ ]:
meses = {
    'jan': 1,
    'fev': 2,
    'mar': 3,
    'abr': 4,
    'mai': 5,
    'jun': 6,
    'jul': 7,
    'ago': 8,
    'set': 9,
    'out': 10,
    'nov': 11,
    'dez': 12,
}

caminho_bases = pathlib.Path('dataset')

# Lista para guardar os DataFrames temporariamente
lista_dfs = []

for arquivo in caminho_bases.iterdir():
    # Filtra para ler apenas arquivos com extensão .csv
    nome_mes = arquivo.name[:3]
    mes = meses[nome_mes]
    ano = arquivo.name[-8:] 
    ano = int(ano.replace('.csv', ''))
    if arquivo.suffix == '.csv':
        df = pd.read_csv(arquivo, low_memory=False)
        df['ano'] = ano
        df['mes'] = mes
        lista_dfs.append(df) # .append() da lista do Python funciona normalmente        

# Junta todos os DataFrames da lista em um único DataFrame
base_airbnb = pd.concat(lista_dfs, ignore_index=True)

display(base_airbnb)

- Como temos muitas colunas nosso modelo pode ficar muito lento 
- Além disso uma analise rapida podemos ver que muitas colunas não são necessario
- Por isso vamos excluir algumas colunas de nossa base
- Tipos de colunas que vamos excluir
    1 - IDs, links e informações não relevantes para o modelo.
    2 - colunas repetidas ou extremamente parecidas com outra (que são a mesma informação para o modelo ex data X ano/mês)
    3 - Colunas preenchidas com texto livre - não rodaremos nenhuma análise de palavras ou algo do tipo
    4 - colunas em que todos os valores ou quase todos os valores são iguais

- Para isso, vamos criar um arquivo em excel com os 1000 primeiros registros para uma análise qualitativa


In [ ]:
base_airbnb.head(1000).to_csv('primeiros_registros.csv', sep=';')

### Se tivermos muitas colunas, já vamos identificar quais colunas podemos excluir

In [ ]:
#vimos que nas 1000 primeiras linhas a coluna experiences_offered estava como none
#então vamos analisar essa coluna em todo o df para ver quais os valores que tem
print(base_airbnb['host_acceptance_rate'].value_counts())
#em toda o df esta nome por isso a coluna será excluida
#outras colunas também foram verificadas e cada uma foi decidida o que fazer(host_acceptance_rate, 


In [ ]:
#tem duas colunas que parecem ser iguais, vamos verificar se são iguais mesmo
print((base_airbnb['host_listings_count']==(base_airbnb['host_total_listings_count'])).value_counts())

In [ ]:
print(base_airbnb['square_feet'].isnull().sum())
#888162 valores vazios, então vamos excluir a coluna


In [ ]:
primeiros_registros = pd.read_csv('primeiros_registros_colunas pre filtradas.csv', sep=';')

### Depois da análise qualitativas das colunas, levando em conta os críterios explicados acima ficamos com as seguintes colunas
'host_response_time','host_response_rate','host_is_superhost','host_listings_count','latitude','longitude','property_type','room_type','accommodates','bathrooms','bedrooms','beds','bed_type','amenities','price','security_deposit','cleaning_fee','guests_included','extra_people','minimum_nights','maximum_nights','number_of_reviews','review_scores_rating','review_scores_accuracy','review_scores_cleanliness','review_scores_checkin','review_scores_communication','review_scores_location','review_scores_value','instant_bookable','is_business_travel_ready','cancellation_policy','ano','mes'


In [ ]:
display(primeiros_registros.columns)

In [ ]:
colunas = ['host_response_time','host_response_rate','host_is_superhost','host_listings_count','latitude','longitude','property_type','room_type','accommodates','bathrooms','bedrooms','beds','bed_type','amenities','price','security_deposit','cleaning_fee','guests_included','extra_people','minimum_nights','maximum_nights','number_of_reviews','review_scores_rating','review_scores_accuracy','review_scores_cleanliness','review_scores_checkin','review_scores_communication','review_scores_location','review_scores_value','instant_bookable','is_business_travel_ready','cancellation_policy','ano','mes']
base_airbnb2 = base_airbnb.loc[:, colunas]
display(base_airbnb2)

### Tratar Valores Faltando

- visualizando os dados, percebemos que existe uma grande disparidade em dados faltantes. As colunas com mais de 
300.000 valores NAN foram excluidas das analises.
- Para as outras colunas como temos muitos dados (mais de 900.000), vamos exluir as linhas que contém dados NAN

In [ ]:
print(base_airbnb2.isnull().sum())
#depois do print acima foi decidido que as colunas com mais de 300000 valores nulos seriam excluidas.

for coluna in base_airbnb2.columns:
    if base_airbnb2[coluna].isnull().sum() > 300000:
        base_airbnb2 = base_airbnb2.drop(columns=[coluna])   
 

In [ ]:
base_airbnb2 = base_airbnb2.dropna()
print(base_airbnb2.isnull().sum())
print(base_airbnb2.shape) 


### Verificar Tipos de Dados em cada coluna

In [ ]:
#vamos ver as colunas que sobraram e o tipo de dado de cada uma
#e ver o primeiro registro para ver se os dados estão coerentes
print(base_airbnb2.dtypes)
print('-'*60)
print(base_airbnb2.iloc[0])



- Como price e extra_people estão sendo reconhecidos como STR, ter que mudar para float

In [ ]:
# no caso price e extra_people estão como object, vamos converter para float
base_airbnb2['price'] = base_airbnb2['price'].str.replace('$','').str.replace(',','').astype(np.float32) 
base_airbnb2['extra_people'] = base_airbnb2['extra_people'].str.replace('$','').str.replace(',','').astype(np.float32) 

print(base_airbnb2.dtypes)

### Análise Exploratória e Tratar Outliers

- Vamos basicamente olhar feature por feature para:
    1. Ver a correlação entre as features e decidir se manteremos todas as features que temos.
    2. Excluir outliers (usaremos como regra, valores abaixo de Q1 - 1.5xAmplitude e valores acima de Q3 + 1.5x Amplitude). Amplitude = Q3 - Q1
    3. Confirmar se todas as features que temos fazem realmente sentido para o nosso modelo ou se alguma delas não vai nos ajudar e se devemos excluir
    
- Vamos começar pelas colunas de preço (resultado final que queremos) e de extra_people (também valor monetário). Esses são os valores numéricos contínuos.

- Depois vamos analisar as colunas de valores numéricos discretos (acommodates, bedrooms, guests_included, etc.)

- Por fim, vamos avaliar as colunas de texto e definir quais categorias fazem sentido mantermos ou não.

MAS CUIDADO: não saia excluindo direto outliers, pense exatamente no que você está fazendo. Se não tem um motivo claro para remover o outlier, talvez não seja necessário e pode ser prejudicial para a generalização. Então tem que ter uma balança ai. Claro que você sempre pode testar e ver qual dá o melhor resultado, mas fazer isso para todas as features vai dar muito trabalho.

Ex de análise: Se o objetivo é ajudar a precificar um imóvel que você está querendo disponibilizar, excluir outliers em host_listings_count pode fazer sentido. Agora, se você é uma empresa com uma série de propriedades e quer comparar com outras empresas do tipo também e se posicionar dessa forma, talvez excluir quem tem acima de 6 propriedades tire isso do seu modelo. Pense sempre no seu objetivo

In [ ]:
#ver a correlação entre as features e decidir se manteremos todas as features que temos.
#Features são as colunas do df, e a target é a coluna que queremos prever, no caso price.
#numeric_only=True para pegar somente as colunas numéricas.
#print(base_airbnb2.corr(numeric_only=True))
plt.figure(figsize=(15,10))
plt.title('Correlação entre as features')
sns.heatmap(base_airbnb2.corr(numeric_only=True), annot=True, cmap='Greens')


### Definição de Funções para Análise de OutLiers

 - Vamos definir algumas funções para ajudar na análise de outliers das colunas

In [ ]:
#Excluir outliers (usaremos como regra, valores abaixo de Q1 - 1.5xAmplitude e valores acima de Q3 + 1.5x Amplitude). Amplitude = Q3 - Q1

def limites(coluna):
    q1 = coluna.quantile(0.25)
    q3 = coluna.quantile(0.75)
    amplitude = q3 - q1
    limite_inferior = q1 - 1.5 * amplitude
    limite_superior = q3 + 1.5 * amplitude
    return limite_inferior, limite_superior

print(limites(base_airbnb2['price']))

def excluir_outliers(df, nome_coluna):
    qtde_linhas = df.shape[0]
    limite_inferior, limite_superior = limites(df[nome_coluna])
    df = df[(df[nome_coluna] >= limite_inferior) & (df[nome_coluna] <= limite_superior)]
    linhas_removidas = qtde_linhas - df.shape[0]
    return df, linhas_removidas

In [ ]:
def diagrama_caixa(coluna):
    fig, (ax1, ax2) = plt.subplots(1, 2)
    fig.set_size_inches(15, 5)
    sns.boxplot(x=coluna, ax=ax1)
    ax2.set_xlim(limites(coluna))
    sns.boxplot(x=coluna, ax=ax2)   

def histograma(coluna):
    plt.figure(figsize=(15,5))
    sns.histplot(coluna, bins=40, kde=True)

def grafico_barras(coluna):
    plt.figure(figsize=(15, 5))
    
    # 1. Filtra os valores contados que estão dentro dos limites desejados
    counts = coluna.value_counts().sort_index()
    lim_min, lim_max = limites(coluna)
    counts_filtrado = counts[(counts.index >= lim_min) & (counts.index <= lim_max)]
    
    # 2. Plota apenas as barras dentro do limite
    ax = sns.barplot(x=counts_filtrado.index, y=counts_filtrado.values)    
       
    # 3. Adiciona margem superior para o texto não ser cortado no topo
    ax.set_ylim(top=ax.get_ylim()[1] * 1.15)
    
    # 4. Adiciona os rótulos acima de cada barra
    ax.bar_label(ax.containers[0], padding=3)



### Price

In [ ]:
diagrama_caixa(base_airbnb2['price'])
histograma(base_airbnb2['price'])

Como estamos construindo um modelo para imóveis comuns, acredito que os valores acima do limite superior serão apenas de imóveis de altíssimo luxo, que não é o nosso objetivo principal. Por isso podemos excluir esses outliers

In [ ]:
base_airbnb2, linhas_removidas = excluir_outliers(base_airbnb2, 'price')
print(f'Foram removidas {linhas_removidas} linhas com outliers na coluna price.')

In [ ]:
histograma(base_airbnb2['price'])

### extra_people                

In [ ]:
diagrama_caixa(base_airbnb2['extra_people'])
histograma(base_airbnb2['extra_people'])

vamos excluir os outliers da coluna extra_people talvez depois ver como fica o modelo com esses dados

In [ ]:
#vamos exluir os outliers da coluna extra_people talvez depois ver como fica o modelo com esses dados
base_airbnb2, linhas_removidas = excluir_outliers(base_airbnb2, 'extra_people')
print(f'Foram removidas {linhas_removidas} linhas com outliers na coluna extra_people.')


In [ ]:
histograma(base_airbnb2['extra_people'])

### host_listings_count

In [ ]:
diagrama_caixa(base_airbnb2['host_listings_count'])
grafico_barras(base_airbnb2['host_listings_count'])

podemos excluir os outliers de host_listings_count porque hosts com mais de 6 imoveis no airbnb não é o publico alvo do projeto (imagino que seja imobiliarias ou profissionais que gerenciam imóveis no airbnb)

In [ ]:
base_airbnb2, linhas_removidas = excluir_outliers(base_airbnb2, 'host_listings_count')
print(f'Foram removidas {linhas_removidas} linhas com outliers na coluna host_listings_count.')

diagrama_caixa(base_airbnb2['host_listings_count'])
grafico_barras(base_airbnb2['host_listings_count'])

### accommodates

In [ ]:
diagrama_caixa(base_airbnb2['accommodates'])
grafico_barras(base_airbnb2['accommodates'])

In [ ]:
base_airbnb2, linhas_removidas = excluir_outliers(base_airbnb2, 'accommodates')
print(f'Foram removidas {linhas_removidas} linhas com outliers na coluna accommodates.')

diagrama_caixa(base_airbnb2['accommodates'])
grafico_barras(base_airbnb2['accommodates'])

### bathrooms

In [ ]:
diagrama_caixa(base_airbnb2['bathrooms'])
grafico_barras(base_airbnb2['bathrooms'])

In [ ]:
base_airbnb2, linhas_removidas = excluir_outliers(base_airbnb2, 'bathrooms')
print(f'Foram removidas {linhas_removidas} linhas com outliers na coluna bathrooms.')

diagrama_caixa(base_airbnb2['bathrooms'])
grafico_barras(base_airbnb2['bathrooms'])

### bedrooms

In [ ]:
diagrama_caixa(base_airbnb2['bedrooms'])
grafico_barras(base_airbnb2['bedrooms'])


### beds

In [ ]:
diagrama_caixa(base_airbnb2['beds'])
grafico_barras(base_airbnb2['beds'])

### guests_included

In [ ]:
diagrama_caixa(base_airbnb2['guests_included'])
plt.figure(figsize=(15, 5))
sns.barplot(x=base_airbnb2['guests_included'].value_counts().index, y=base_airbnb2['guests_included'].value_counts().values)
#como neste caso deu uma mensagem de warning, fizemos um gráfico de barras diferente, filtrando os valores que estão dentro dos limites.
#vimos que a maioria dos valores estão em 1, então é possivel que seja uma coluna com erro de preenchimento, mas vamos manter por enquanto, pois pode ser que seja uma coluna que não tenha erro de preenchimento.


Vamos excluir a coluna pois exluir os outliers, pode dar algum problema. Parece que os usuario usam o padrão do arbnb 1. Isso pode levar o nosso modelo uma feature que não é essencial para a definição do preço. Por isso parece  melhor exluir a coluna da analise

In [ ]:
base_airbnb2 = base_airbnb2.drop(columns=['guests_included'])


In [ ]:
print(base_airbnb2.shape)

### minimum_nights

In [ ]:
diagrama_caixa(base_airbnb2['minimum_nights'])
grafico_barras(base_airbnb2['minimum_nights'])

In [ ]:
base_airbnb2, linhas_removidas = excluir_outliers(base_airbnb2, 'minimum_nights')
print(f'Foram removidas {linhas_removidas} linhas com outliers na coluna minimum_nights.')

diagrama_caixa(base_airbnb2['minimum_nights'])
grafico_barras(base_airbnb2['minimum_nights'])

### maximum_nights

In [ ]:
diagrama_caixa(base_airbnb2['maximum_nights'])
grafico_barras(base_airbnb2['maximum_nights'])

maximum_nights tambem parece ter outliers, prenchidos com valores padrões 0 e também com valores muito altos, então vamos exluir os outliers da coluna maximum_nights talvez depois ver como fica o modelo com esses dados

In [ ]:
base_airbnb2 = base_airbnb2.drop(columns=['maximum_nights'])
print(base_airbnb2.shape)

### number_of_reviews

In [ ]:
diagrama_caixa(base_airbnb2['number_of_reviews'])
grafico_barras(base_airbnb2['number_of_reviews'])

sera deixado os reviews pois se tirar os que tem mais será retirado os mais antigos que tem mais tempo de arbnb.
então a coluna será retirada pois os que não tem reviews são novos e podem querer estar disponibilizando para aluguel

In [ ]:
base_airbnb2 = base_airbnb2.drop(columns=['number_of_reviews'])
print(base_airbnb2.shape)

### Tratamento de colunas com valores de texto

- property_type

In [ ]:
print(base_airbnb2['property_type'].value_counts())
plt.figure(figsize=(15, 5))
grafico = sns.countplot(x='property_type', data=base_airbnb2)
grafico.tick_params(axis='x', rotation=90)

In [ ]:
tabela_tipos_casa = base_airbnb2['property_type'].value_counts()
colunas_agrupar = []
for tipo in tabela_tipos_casa.index:
    if tabela_tipos_casa[tipo] < 2000:
        colunas_agrupar.append(tipo)

for tipo in colunas_agrupar:
    base_airbnb2.loc[base_airbnb2['property_type'] == tipo, 'property_type'] = 'Outros'

print(base_airbnb2['property_type'].value_counts())

plt.figure(figsize=(15, 5))
grafico = sns.countplot(x='property_type', data=base_airbnb2)
grafico.tick_params(axis='x', rotation=90)


- room_type

In [ ]:
print(base_airbnb2['room_type'].value_counts())
plt.figure(figsize=(15, 5))
grafico = sns.countplot(x='room_type', data=base_airbnb2)
grafico.tick_params(axis='x', rotation=90)
#não será necessário agrupar os tipos de room_type, pois são poucos tipos e a quantidade de cada tipo é grande.

- bed_type                        

In [ ]:
print(base_airbnb2['bed_type'].value_counts())
plt.figure(figsize=(15, 5))
grafico = sns.countplot(x='bed_type', data=base_airbnb2)
grafico.tick_params(axis='x', rotation=90)             


In [ ]:
#aqui será necessário agrupar porque a quantidadede valores de alguns são muito pequenos, 
# então vamos agrupar em Outros tipos de camas com valores menres que 10000 
tabela_bed_type = base_airbnb2['bed_type'].value_counts()
colunas_agrupar = []
for tipo in tabela_bed_type.index:
    if tabela_bed_type[tipo] < 10000:
        colunas_agrupar.append(tipo)

for tipo in colunas_agrupar:
    base_airbnb2.loc[base_airbnb2['bed_type'] == tipo, 'bed_type'] = 'Outros'

print(base_airbnb2['bed_type'].value_counts())

plt.figure(figsize=(15, 5))
grafico = sns.countplot(x='bed_type', data=base_airbnb2)
grafico.tick_params(axis='x', rotation=90)

- cancellation_policy

In [ ]:
print(base_airbnb2['cancellation_policy'].value_counts())
plt.figure(figsize=(15, 5))
grafico = sns.countplot(x='cancellation_policy', data=base_airbnb2)
grafico.tick_params(axis='x', rotation=90)

In [ ]:
#aqui será necessário agrupar porque a quantidade valores de alguns são muito pequenos, 
# então vamos agrupar em strict com valores menores que 10000 
tabela_cancellation = base_airbnb2['cancellation_policy'].value_counts()
colunas_agrupar = []
for tipo in tabela_cancellation.index:
    if tabela_cancellation[tipo] < 10000:
        colunas_agrupar.append(tipo)

for tipo in colunas_agrupar:
    base_airbnb2.loc[base_airbnb2['cancellation_policy'] == tipo, 'cancellation_policy'] = 'strict'

print(base_airbnb2['cancellation_policy'].value_counts())

plt.figure(figsize=(15, 5))
grafico = sns.countplot(x='cancellation_policy', data=base_airbnb2)
grafico.tick_params(axis='x', rotation=90)

- amenities         
como temos uma diversidade muito grande de amenities e as vezes a mesmas amenities pode ser escritas de formas diferentes vamos avaliar a quatidade de amenities como parametro para o nosso modelo



In [ ]:
# na coluna os itens estão separados por ',' então vamos separar tudo por virgula e contar os itens

print(base_airbnb2['amenities'].iloc[0].split(','))
print(len(base_airbnb2['amenities'].iloc[0].split(',')))

#criar uma coluna com o numero dos amenities e depois excluir a coluna amenities

base_airbnb2['qtde_amenities'] = base_airbnb2['amenities'].str.split(',').apply(len)

base_airbnb2 = base_airbnb2.drop(columns=['amenities'])
display(base_airbnb2)



In [ ]:
diagrama_caixa(base_airbnb2['qtde_amenities'])
grafico_barras(base_airbnb2['qtde_amenities'])

In [ ]:
base_airbnb2, linhas_removidas = excluir_outliers(base_airbnb2, 'qtde_amenities')
print(f'Foram removidas {linhas_removidas} linhas com outliers na coluna qtde_amenities.')

diagrama_caixa(base_airbnb2['qtde_amenities'])
grafico_barras(base_airbnb2['qtde_amenities'])

### Visualização de mapa das propriedades

In [ ]:
amostra = base_airbnb2.sample(n=50000)

centro_mapa = {'lat': amostra.latitude.mean(), 'lon': amostra.longitude.mean()}

mapa = px.density_map(
    amostra, lat='latitude', lon='longitude', z='price',
    radius=1,
    center=centro_mapa, zoom=10,
    map_style="open-street-map",
    #map_style="carto-positron",
    color_continuous_scale="Plasma",    
    opacity=0.8,)
mapa.show()

### Encoding
Precisamos juntar as features para facilitar o trabalho do modelo futuro (features de categorias true e false etc..)
- Features true e false vamos substituir por True por 1 e False por 0
- Features de categoria (fatures que os valores da colunas são textos) vamos utilizar o método de encoding de variáveis dummies


In [ ]:
colunas_tf = ['host_is_superhost', 'instant_bookable', 'is_business_travel_ready']
base_airbnb_cod = base_airbnb2.copy()
#print(base_airbnb_cod.iloc[0])    
#print(base_airbnb_cod.dtypes)
for coluna in colunas_tf:
    #base_airbnb_cod.loc[linha, coluna]
    base_airbnb_cod.loc[base_airbnb_cod[coluna] == 't', coluna] = '1'
    base_airbnb_cod.loc[base_airbnb_cod[coluna] == 'f', coluna] = '0'

#print(base_airbnb_cod.iloc[0])    

#print(base_airbnb_cod.dtypes)



In [ ]:
colunas_categorias = ['property_type', 'room_type', 'bed_type','cancellation_policy']
#pd.get_dummies(dataframe, colunas)
base_airbnb_cod = pd.get_dummies(data=base_airbnb_cod, columns=colunas_categorias)
display(base_airbnb_cod.head())
print(base_airbnb_cod.dtypes)

### Modelo de Previsão

- Métricas de avaliação

In [ ]:
def avaliar_modelo(nome_modelo, y_teste, previsao):
    r2 = r2_score(y_teste, previsao)
    rmse = np.sqrt(mean_squared_error(y_teste, previsao))    
    return f'Métricas do modelo {nome_modelo}: \nR²: {r2:.2%} \nRMSE: {rmse:.2f}'



- Escolha dos modelos a serem testados
    1. RandomForest
    2. LinearRegression
    3. Extra Tree
- Separação das Variáveis

In [ ]:
modelo_rf = RandomForestRegressor()
modelo_lr = LinearRegression()
modelo_et = ExtraTreesRegressor()

modelos = {'RandomForest': modelo_rf,
           'LinearRegression': modelo_lr,
           'ExtraTrees': modelo_et}

y = base_airbnb_cod['price']
x = base_airbnb_cod.drop(columns=['price'])

- Separar os dados em treino e teste + treino do modelo

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, random_state=10)
#o parametro random_state é usado para garantir que a divisão dos dados seja reproduzível.
#Isso significa que, se você executar o código várias vezes com o mesmo valor de random_state,
# obterá a mesma divisão dos dados em conjuntos de treino e teste.

for nome_modelo, modelo in modelos.items():
    #treinar o modelo
    tempo_inicio = time.time()
    modelo.fit(x_train, y_train)
    #testar o modelo
    previsao = modelo.predict(x_test)
    print(avaliar_modelo(nome_modelo, y_test, previsao)) 
    tempo_fim = time.time()
    tempo_execucao = tempo_fim - tempo_inicio
    print(f'{nome_modelo}: Tempo de execução do treino e teste: {tempo_execucao:.2f} segundos')
    print('-'*60)


### Análise do Melhor Modelo

In [ ]:
for nome_modelo, modelo in modelos.items():    
    #testar o modelo
    hora_inicio = time.time()
    previsao = modelo.predict(x_test)        
    print(avaliar_modelo(nome_modelo, y_test, previsao)) 
    hora_fim = time.time()
    tempo_execucao = hora_fim - hora_inicio    
    print(f'{nome_modelo}: Tempo de execução do teste: {tempo_execucao:.2f} segundos')
    print('-'*60)

- Modelo escolhido como melhor: ExtraTreesRegressor
esse foi o modelo com maior R² e menor RMSE, e como não tivemos uma grande diferença de velocidade de treino entre os modelo RandomForest (que é o que teve os resultados mais próximos do ExtraTreesRegressor) e o ExtraTreesRegressor, optamos por escolher o ExtraTreesRegressor como modelo final. 

- O modelo de regressão linear não obteve um resultado satisfatório com valores muito pior que os outros modelos

### Ajustes e Melhorias no Melhor Modelo

In [ ]:
#mostrar as importâncias das features do modelo ExtraTreesRegressor
#print(modelo_et.feature_importances_)
#print(x_train.columns)

df_importancias_features = pd.DataFrame(modelo_et.feature_importances_, x_train.columns)
df_importancias_features = df_importancias_features.sort_values(by=0, ascending=False)
display(df_importancias_features)
plt.figure(figsize=(15, 5))
ax = sns.barplot(x= df_importancias_features.index, y= df_importancias_features[0])
ax.tick_params(axis='x', rotation=90)
#colocar os valores das importâncias acima das barras com duas casas decimais
for p in ax.patches:
    ax.annotate(f'{p.get_height():.2f}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', fontsize=10, color='black', xytext=(0, 5),
                textcoords='offset points')

# Ajuste final do modelo 

- Verificado que algumas features não são tão importantes para o modelo, então vamos excluir as features com importância novamente o modelo ExtraTreesRegressor.
- Resultado das métricas do modelo vencedor 

Métricas do modelo ExtraTrees: 
- R²: 97.45% 
- RMSE: 42.62
- ExtraTrees: Tempo de execução do treino e teste: 734.04 segundos


- is_business_travel_ready não parece relevante para o modelo

In [ ]:
#retirar a feature 'is_business_travel_ready'
base_airbnb_cod = base_airbnb_cod.drop(columns=['is_business_travel_ready'])
y = base_airbnb_cod['price']
x = base_airbnb_cod.drop(columns=['price'])
x_train, x_test, y_train, y_test = train_test_split(x, y, random_state=10)
#treinar o modelo ExtraTreesRegressor novamente com a feature retirada
nome_modelo = 'ExtraTrees'
tempo_inicio = time.time()
modelo_et.fit(x_train, y_train)
#testar o modelo
previsao = modelo_et.predict(x_test)
print(avaliar_modelo(nome_modelo, y_test, previsao)) 
tempo_fim = time.time()
tempo_execucao = tempo_fim - tempo_inicio
print(f'{nome_modelo}: Tempo de execução do treino e teste: {tempo_execucao:.2f} segundos')
print('-'*60)

In [ ]:
#vamos testar o modelo retirando bed_type_Real e bed_type_outros
base_teste = base_airbnb_cod.copy()

#print(base_teste.shape)
for coluna in base_teste:
    if 'bed_type' in coluna:
        base_teste = base_teste.drop(columns=[coluna])
#print(base_teste.columns)        
#print(base_teste.shape)

y = base_teste['price']
x = base_teste.drop(columns=['price'])
x_train, x_test, y_train, y_test = train_test_split(x, y, random_state=10)
#treinar o mopdelo ExtraTreesRegressor novamente com a feature retirada
nome_modelo = 'ExtraTrees'
tempo_inicio = time.time()
modelo_et.fit(x_train, y_train)
#testar o modelo
previsao = modelo_et.predict(x_test)
print(avaliar_modelo(nome_modelo, y_test, previsao)) 
tempo_fim = time.time()
tempo_execucao = tempo_fim - tempo_inicio
print(f'{nome_modelo}: Tempo de execução do treino e teste: {tempo_execucao:.2f} segundos')
print('-'*60)

#verificado os dados apos a retirada das features bed_type_Real e bed_type_outros,
# o modelo ficou com um desempenho bem parecidos então vamos deixar sem as duas features no modelo, 
#pois elas não foram importantes para o modelo, mesmo que o desempenho não tenha melhorado muito.

In [ ]:
print(base_teste.columns) 

# Deploy do Projeto

- Passo 1 -> Criar arquivo do Modelo (joblib)<br>
- Passo 2 -> Escolher a forma de deploy:
    - Arquivo Executável + Tkinter
    - Deploy em Microsite (Flask)
    - Deploy apenas para uso direto Streamlit
- Passo 3 -> Outro arquivo Python (pode ser Jupyter ou PyCharm)
- Passo 4 -> Importar streamlit e criar código
- Passo 5 -> Atribuir ao botão o carregamento do modelo
- Passo 6 -> Deploy feito

In [ ]:
x['price'] = y
x.to_csv('dados.csv')

In [ ]:
import joblib
joblib.dump(modelo_et, 'modelo.joblib')
